In [60]:
import pandas as pd
import numpy as np
from tabulate import tabulate
import sys
from IPython.core.interactiveshell import InteractiveShell
import holidays
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import LabelEncoder
import shap

def setup_korean_font():
    """matplotlib 한글 폰트 설정"""
    font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    
    if os.path.exists(font_path):
        font_prop = fm.FontProperties(fname=font_path)
        font_name = font_prop.get_name()
        plt.rcParams['font.family'] = font_name
        plt.rcParams['axes.unicode_minus'] = False
        print(f"한글 폰트 설정 완료: {font_name}")
        return True
    else:
        print("나눔고딕 폰트를 찾을 수 없습니다.")
        return False

# 폰트 설정 실행
setup_korean_font()
pd.set_option('display.max_columns', None)

한글 폰트 설정 완료: NanumGothic


# 데이터 확인  

In [61]:
def check_data(path):
    df = pd.read_csv(path, encoding='utf-8')
    print('데이터 컬럼 확인')
    print(df.columns, '\n')
    print('데이터 상위 5개 추출')
    print(tabulate(df.head(),headers=df.columns), '\n')
    print('데이터 타입 확인')
    print(df.info())
    print('데이터 Nan값 개수 확인')
    print(df.isna().sum())


## building_info 데이터 확인  

- 건물번호: 각 건물의 고유 식별번호  
- 건물유형: 건물의 용도별 분류 (호텔, 상용, 병원, 학교 등)  
- 연면적(m²): 건물 전체 바닥면적 (m2)  
- 냉방면적(m²): 공조시스템이 설치된 면적 (m2)  
- 태양광용량(kW): 설치된 태양광 발전 시설의 용량 (kWh)  
- ESS저장용량(kWh): 에너지 저장 시스템(Energy Storage System) 용량 (kWh)  
- PCS용량(kW): 전력변환시스템(Power Conversion System) 용량 (kW)  

In [62]:
data_path = 'datas/building_info.csv' 
check_data(data_path)
building_info_df = pd.read_csv(data_path)
building_info_df.head().to_csv('temp.csv', encoding='utf-8')

데이터 컬럼 확인
Index(['건물번호', '건물유형', '연면적(m2)', '냉방면적(m2)', '태양광용량(kW)', 'ESS저장용량(kWh)',
       'PCS용량(kW)'],
      dtype='object') 

데이터 상위 5개 추출
      건물번호  건물유형      연면적(m2)    냉방면적(m2)  태양광용량(kW)    ESS저장용량(kWh)    PCS용량(kW)
--  ----------  ----------  ------------  --------------  ----------------  ------------------  -------------
 0           1  호텔             82912.7         77586    -                 -                   -
 1           2  상용             40658.9         30392.8  -                 -                   -
 2           3  병원            560431          418992    278.58            -                   -
 3           4  호텔             41813.3         23715.7  -                 -                   -
 4           5  학교            403749          248507    1983.05           1025                250 

데이터 타입 확인
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        ------

## train 데이터 확인  

- num_date_time: 고유 식별자 (건물번호_날짜시간 형식)  
- 건물번호: 건물 식별 번호  
- 일시: 측정 날짜와 시간 (YYYYMMDD HH 형식)  
- 기온: 외부 기온 (섭씨)  
- 강수량: 시간당 강수량   
- 풍속: 풍속   
- 습도: 상대습도   
- 일조: 일조시간   
- 일사(MJ/m²): 일사량   
- 전력소비량(kWh): 건물의 시간별 전력 소비량 (kWh)  

In [63]:
data_path = 'datas/train.csv'
check_data(data_path)
train_df = pd.read_csv(data_path)
print(train_df.shape)

데이터 컬럼 확인
Index(['num_date_time', '건물번호', '일시', '기온(°C)', '강수량(mm)', '풍속(m/s)', '습도(%)',
       '일조(hr)', '일사(MJ/m2)', '전력소비량(kWh)'],
      dtype='object') 

데이터 상위 5개 추출
    num_date_time      건물번호  일시           기온(°C)    강수량(mm)    풍속(m/s)    습도(%)    일조(hr)    일사(MJ/m2)    전력소비량(kWh)
--  ---------------  ----------  -----------  ----------  ------------  -----------  ---------  ----------  -------------  -----------------
 0  1_20240601 00             1  20240601 00        18.3             0          2.6         82           0              0            5794.8
 1  1_20240601 01             1  20240601 01        18.3             0          2.7         82           0              0            5591.85
 2  1_20240601 02             1  20240601 02        18.1             0          2.6         80           0              0            5338.17
 3  1_20240601 03             1  20240601 03        18               0          2.6         81           0              0            4554.42
 4  1_20

In [64]:
# 열 수정
train_df.columns = ['num_date_time', '건물번호', '일시', '기온', '강수량', '풍속', '습도', '일조', '일사', '전력소비량']

## 연, 월, 일, 시간 추출  

In [65]:
# 일시에서 날짜와 시간 추출
train_df['시간'] = train_df['일시'].str.split(' ').str[1]
train_df['날짜'] = train_df['일시'].str.split(' ').str[0]
train_df['날짜'] = pd.to_datetime(train_df['날짜'], format='%Y%m%d')

train_df['연'] = train_df['날짜'].dt.year
train_df['월'] = train_df['날짜'].dt.month
train_df['일'] = train_df['날짜'].dt.day
train_df.head()

,num_date_time,건물번호,일시,기온,강수량,풍속,습도,일조,일사,전력소비량,시간,날짜,연,월,일
0,1_20240601 00,1,20240601 00,18.3,0.0,2.6,82.0,0.0,0.0,5794.80,00,2024-06-01,2024,6,1
1,1_20240601 01,1,20240601 01,18.3,0.0,2.7,82.0,0.0,0.0,5591.85,01,2024-06-01,2024,6,1
2,1_20240601 02,1,20240601 02,18.1,0.0,2.6,80.0,0.0,0.0,5338.17,02,2024-06-01,2024,6,1
3,1_20240601 03,1,20240601 03,18.0,0.0,2.6,81.0,0.0,0.0,4554.42,03,2024-06-01,2024,6,1
4,1_20240601 04,1,20240601 04,17.8,0.0,1.3,81.0,0.0,0.0,3602.25,04,2024-06-01,2024,6,1


## 계절 생성  

In [66]:
# 계절 생성
condition = [
    train_df['월'].isin([3,4,5]),
    train_df['월'].isin([6,7,8]),
    train_df['월'].isin([9,10,11]),
    train_df['월'].isin([12,1,2])
]

choice  = ['0', '1', '2', '3']
train_df['계절'] = np.select(condition, choice, default='알수없음')
train_df['계절'] = train_df['계절'].astype(int)
train_df.head()

,num_date_time,건물번호,일시,기온,강수량,풍속,습도,일조,일사,전력소비량,시간,날짜,연,월,일,계절
0,1_20240601 00,1,20240601 00,18.3,0.0,2.6,82.0,0.0,0.0,5794.80,00,2024-06-01,2024,6,1,1
1,1_20240601 01,1,20240601 01,18.3,0.0,2.7,82.0,0.0,0.0,5591.85,01,2024-06-01,2024,6,1,1
2,1_20240601 02,1,20240601 02,18.1,0.0,2.6,80.0,0.0,0.0,5338.17,02,2024-06-01,2024,6,1,1
3,1_20240601 03,1,20240601 03,18.0,0.0,2.6,81.0,0.0,0.0,4554.42,03,2024-06-01,2024,6,1,1
4,1_20240601 04,1,20240601 04,17.8,0.0,1.3,81.0,0.0,0.0,3602.25,04,2024-06-01,2024,6,1,1


## 요일 및 휴일 추가  

* 공휴일, 주말 = 1  
* 평일 = 0  

월요일: 0  
...  
일요일: 6  

In [69]:
# 요일 추가
train_df['요일'] = train_df['날짜'].dt.weekday # 요일 번호 (0=월요일, 6=일요일)

# 한국 공휴일 객체 생성 (year 지정 가능)
kr_holidays = holidays.KR()

# 휴일: 1, 평일: 0
train_df['휴일'] = train_df['날짜'].apply(lambda x: 1 if x in kr_holidays else 0)
train_df['휴일'] = train_df.apply(lambda x: 1 if x['요일'] >= 5 else x['휴일'], axis=1)

train_df.head()

,num_date_time,건물번호,일시,기온,강수량,풍속,습도,일조,일사,전력소비량,시간,날짜,연,월,일,계절,요일,휴일
0,1_20240601 00,1,20240601 00,18.3,0.0,2.6,82.0,0.0,0.0,5794.80,00,2024-06-01,2024,6,1,1,5,1
1,1_20240601 01,1,20240601 01,18.3,0.0,2.7,82.0,0.0,0.0,5591.85,01,2024-06-01,2024,6,1,1,5,1
2,1_20240601 02,1,20240601 02,18.1,0.0,2.6,80.0,0.0,0.0,5338.17,02,2024-06-01,2024,6,1,1,5,1
3,1_20240601 03,1,20240601 03,18.0,0.0,2.6,81.0,0.0,0.0,4554.42,03,2024-06-01,2024,6,1,1,5,1
4,1_20240601 04,1,20240601 04,17.8,0.0,1.3,81.0,0.0,0.0,3602.25,04,2024-06-01,2024,6,1,1,5,1


## 시간대 만들기  

6 ~ 9 : 출근 시간(0)  
9 ~ 18 : 근무 시간(1)  
18 ~ 21 : 퇴근 시간(2)  
21 ~ 6 : 새벽 시간(3)  

In [74]:
def get_time_period(hour):
    if 6 <= hour < 9:
        return 0
    elif 9 <= hour < 18:
        return 1
    elif 18 <= hour < 21:
        return 2
    else:
        return 3

train_df['시간대'] = train_df['시간'].astype(int).apply(get_time_period)
train_df.head()

,num_date_time,건물번호,일시,기온,강수량,풍속,습도,일조,일사,전력소비량,시간,날짜,연,월,일,계절,요일,휴일,시간대
0,1_20240601 00,1,20240601 00,18.3,0.0,2.6,82.0,0.0,0.0,5794.80,00,2024-06-01,2024,6,1,1,5,1,3
1,1_20240601 01,1,20240601 01,18.3,0.0,2.7,82.0,0.0,0.0,5591.85,01,2024-06-01,2024,6,1,1,5,1,3
2,1_20240601 02,1,20240601 02,18.1,0.0,2.6,80.0,0.0,0.0,5338.17,02,2024-06-01,2024,6,1,1,5,1,3
3,1_20240601 03,1,20240601 03,18.0,0.0,2.6,81.0,0.0,0.0,4554.42,03,2024-06-01,2024,6,1,1,5,1,3
4,1_20240601 04,1,20240601 04,17.8,0.0,1.3,81.0,0.0,0.0,3602.25,04,2024-06-01,2024,6,1,1,5,1,3


## 체감온도, 불쾌지수, 강수여부   

$$  
체감온도 = 기온 - (풍속 \times 0.5)  
$$  
$$  
불쾌지수 =  0.81 \times 기온 + 0.01 \times 습도 \times (0.99 \times 기온 - 14.3) + 46.3  
$$  
$$  
강수여부 = 강수량 > 0  
$$  

In [81]:
train_df['체감온도'] = train_df['기온'] - (train_df['풍속'] * 0.5)
train_df['불쾌지수'] = round(0.81 * train_df['기온'] + 0.01 * train_df['습도'] * (0.99 * train_df['기온'] - 14.3) + 46.3, 2)
train_df['강수여부'] = train_df['강수량'].apply(lambda x: 1 if x > 0 else 0)

train_df.head()

,num_date_time,건물번호,일시,기온,강수량,풍속,습도,일조,일사,전력소비량,시간,날짜,연,월,일,계절,요일,휴일,시간대,체감온도,불쾌지수,강수여부
0,1_20240601 00,1,20240601 00,18.3,0.0,2.6,82.0,0.0,0.0,5794.80,00,2024-06-01,2024,6,1,1,5,1,3,17.00,64.25,0
1,1_20240601 01,1,20240601 01,18.3,0.0,2.7,82.0,0.0,0.0,5591.85,01,2024-06-01,2024,6,1,1,5,1,3,16.95,64.25,0
2,1_20240601 02,1,20240601 02,18.1,0.0,2.6,80.0,0.0,0.0,5338.17,02,2024-06-01,2024,6,1,1,5,1,3,16.80,63.86,0
3,1_20240601 03,1,20240601 03,18.0,0.0,2.6,81.0,0.0,0.0,4554.42,03,2024-06-01,2024,6,1,1,5,1,3,16.70,63.73,0
4,1_20240601 04,1,20240601 04,17.8,0.0,1.3,81.0,0.0,0.0,3602.25,04,2024-06-01,2024,6,1,1,5,1,3,17.15,63.41,0


## 온도구간 설정  

10도 미만: 추움(0)  
20도 미만: 쌀쌀함(1)  
25도 미만: 적당함(2)  
30도 미만: 따뜻함(3)  
30도 이상: 더움(4)  

In [83]:
# 6. 온도 구간별 분류
def temp_category(temp):
    if temp < 10:
        return 0
    elif temp < 20:
        return 1
    elif temp < 25:
        return 2
    elif temp < 30:
        return 3
    else:
        return 4

train_df['온도구간'] = train_df['기온'].apply(temp_category)

train_df.head()

,num_date_time,건물번호,일시,기온,강수량,풍속,습도,일조,일사,전력소비량,시간,날짜,연,월,일,계절,요일,휴일,시간대,체감온도,불쾌지수,강수여부,온도구간
0,1_20240601 00,1,20240601 00,18.3,0.0,2.6,82.0,0.0,0.0,5794.80,00,2024-06-01,2024,6,1,1,5,1,3,17.00,64.25,0,1
1,1_20240601 01,1,20240601 01,18.3,0.0,2.7,82.0,0.0,0.0,5591.85,01,2024-06-01,2024,6,1,1,5,1,3,16.95,64.25,0,1
2,1_20240601 02,1,20240601 02,18.1,0.0,2.6,80.0,0.0,0.0,5338.17,02,2024-06-01,2024,6,1,1,5,1,3,16.80,63.86,0,1
3,1_20240601 03,1,20240601 03,18.0,0.0,2.6,81.0,0.0,0.0,4554.42,03,2024-06-01,2024,6,1,1,5,1,3,16.70,63.73,0,1
4,1_20240601 04,1,20240601 04,17.8,0.0,1.3,81.0,0.0,0.0,3602.25,04,2024-06-01,2024,6,1,1,5,1,3,17.15,63.41,0,1
